In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import os

In [2]:
current_dir = Path(os.getcwd())
try:
    project_root = next(p for p in current_dir.parents if (p / 'src').exists() or p.name == 'PedSimCity')
except StopIteration:
    project_root = current_dir

# Define the source file path relative to the project root
file_path = project_root / "src" / "main" / "resources" / "TorinoCentre" / "Torino_censusData.gpkg"

try:
    # Read the GeoPackage file
    gdf = gpd.read_file(file_path)

    # Inspect and Preview
    print(f"Successfully loaded file from: {file_path}")
    print("\nColumns available:", gdf.columns.tolist())
    print(f"Coordinate Reference System: {gdf.crs}")
    print("\nFirst 5 rows:")
    print(gdf.head())

except Exception as e:
    print(f"Error reading the file: {e}")

Successfully loaded file from: c:\Users\sgabalog\Documents\PedSim\Working_Version_Connected_to_Gab\PedSimCity\src\main\resources\TorinoCentre\Torino_censusData.gpkg

Columns available: ['COD_REG', 'COD_UTS', 'PRO_COM', 'SEZ21', 'censusZoneID', 'censusZoneTypeID', 'areaType', 'areaID', 'COM_ASC1', 'COM_ASC2', 'COM_ASC3', 'population', 'families', 'residentialUnits', 'residentialBuildings', 'PROCOM', 'SEZ21_ID', 'P1', 'P2', 'P3', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P36', 'P37', 'P38', 'P39', 'P40', 'P41', 'P42', 'P43', 'P44', 'P45', 'P67', 'P68', 'P69', 'P70', 'P71', 'P72', 'P73', 'P74', 'P75', 'P76', 'P77', 'P78', 'P79', 'P80', 'P81', 'P82', 'P83', 'P84', 'P85', 'P86', 'P87', 'P88', 'P89', 'P90', 'P91', 'P92', 'P93', 'P94', 'P95', 'P96', 'P97', 'P98', 'P99', 'P100', 'P101', 'P102', 'P103', 'IT1', 'IT2', 'IT3', 'IT4', 'IT5', 'IT6', 'IT7', 'IT8', 'IT9', 'IT10', 'IT11', 'I

In [7]:
# Calculate the % of vulnerable individuals within each zone
# (Total Females + Males 65+) / Total Population
gdf['vulnerability_pct'] = (gdf['P3'] + gdf['P19']) / gdf['P1']
gdf['vulnerability_pct'] = gdf['vulnerability_pct'].fillna(0)

# 3. Optional: Ensure the probability never exceeds 100% (safety clip)
gdf['vulnerability_pct'] = gdf['vulnerability_pct'].clip(0, 1)

# Preview the results
print(gdf[['SEZ21_ID', 'P1', 'vulnerability_pct']].head())

       SEZ21_ID    P1  vulnerability_pct
0           NaN   NaN                0.0
1           NaN   NaN                0.0
2           NaN   NaN                0.0
3  1.272001e+10  20.0                0.6
4  1.272001e+10   0.0                0.0


In [9]:
# Remove all rows where population is 0
# This removes parks, industrial zones, and uninhabited areas
gdf = gdf[gdf['P1'] > 0].copy()

In [3]:
# 1. Define the columns to keep
columns_to_save = ['SEZ21_ID', 'vulnerability_pct', 'geometry']

# 2. Filter the GeoDataFrame
gdf_vulnerability = gdf[columns_to_save].copy()

# 3. Define the output path relative to the project root
output_path = project_root / "src" / "main" / "resources" / "TorinoCentre" / "TorinoCentre_censusData_vulnerability.gpkg"

try:
    # 4. Save to GPKG
    # Ensure the directory exists before saving
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    gdf_vulnerability.to_file(output_path, layer='vulnerability_weights', driver="GPKG")
    print(f"Successfully saved vulnerability data to: {output_path}")

except Exception as e:
    print(f"An error occurred while saving: {e}")

KeyError: "['vulnerability_pct'] not in index"